<a href="https://colab.research.google.com/github/Basty241/SteelPlateFaultsWithDatabricks/blob/main/Entrega_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#PRESENTACIÓN 1: Steel Plates Faults
Integrantes:

- Bastián Araya

- Melisa Gónzalez

#1 Introducción
*Escribe una breve descripción de en qué consiste el Data set, sus variables y cuál es el target.
(UCI Machine Learning Repository / Semeion Research Center, Buscema et al., 2010)


#2 Cargar el problema

In [0]:
%pip install seaborn
%restart_python

###2.1 Importaciones

In [0]:
# Pandas es utilizado para leer los set de datos
import pandas as pd

# Para generar series de datos a graficar.
import numpy as np

# Para generar gráficos
from matplotlib import pyplot as plt
import seaborn as sns

# Para estadística inferencial y sets de hipótesis
import statistics as st
from scipy import stats

###2.2 Archivos

In [0]:
# 1. Ruta del dataset en el volumen de Unity Catalog
ruta_csv = "/Volumes/workspace/default/steel-plate-faults/faults.csv"

# 2. Cargar el dataset completo en un dataframe con Pandas.
df = pd.read_csv(ruta_csv)

###2.3 Funciones

In [0]:
# Ordenar dataframe
def contarOrdenar(df1, columna):
    conteo = df1[columna].value_counts()
    conteo_ordenado = conteo.sort_index(ascending=True) # Ordena cada longitud de cinta de menor a mayor
    return conteo_ordenado

# Extraer outliers
def extraer_outliers_iqr(df_in, cols):
  filas_outliers = []
  for col in cols:
    q1 = df_in[col].quantile(0.25)
    q3 = df_in[col].quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr

    outliersS = df_in[df_in[col]>lim_sup]
    outliersI = df_in[df_in[col]<lim_inf]

    outliers = pd.concat([outliersI, outliersS])
    filas_outliers.append(outliers)

  return filas_outliers


#3 Investigación de Estructura y Calidad del Dataset

##3.1 Investigación de Estructura
Vamos a utilizar métodos de pandas para obtener información sobre la estructura del dataset.

In [0]:
# Visualizar aspectos básicos del dataset:
## Visualización de las primeras 3 líneas: ocupando .head(3)
encabezado = df.head(3)
print(encabezado, '\n')

## Visualización de las primeras 3 líneas: ocupando .head(3)
cola = df.tail(3)
print(cola, '\n')

## Tamaño del data set: ocupando .shape
filaColumna = df.shape
print(filaColumna, '\n')

## Información del tipo de datos: ocupando .info()
infoDataframe = df.info()

Con df.info() validamos que no hay datos nulos en toda la tabla, que estamos trabajando con las variables correctas y el tipo de dato que se le asigna a cada variable.
Además, del número de filas (1941) y columnas (34) del dataset.
Como K_Scatch está mal escrito lo vamos a cambiar a K_Scratch.

In [0]:
# Corregimos nombre de la columna K_Scatch
df = df.rename(columns={'K_Scatch': 'K_Scratch'})
df.info()

## 3.2 Separación de columnas
Para poder determinar correctamente el tipo de falla, separamos las variables por categoria.
- Coordenadas cartesianas: Delimitan la posición de la falla en la placa.
- Geometría: Área, proyección del perímetro en los ejes y espesor.
- Luminosidad: Intensidad lumínica en los pixeles de la falla.
- Longitud de Cinta: Variable operativa del transporte de las placas.
- Tipo de Acero: A300 o A400.
- Índices: Permiten describir la falla en relación a distintas variables, como reflectividad de la placa, proximidad a los bordes, orientación de la falla, forma, etc.
- Tipo de Falla (Target): Corresponde a la variable que queremos clasificar en función de las otras variables. Para posterior automatización de detección.

In [0]:
# Separación de columnas para análisis.
## Extrae los nombres de las columnas.
columnas = list(df.columns)

## Separamos por tipo de dato.
coordenadas = columnas[0:4]

geometria = columnas[4:7] + [columnas[13]]

luminosidad = columnas[7:10]

longitudDeCinta = columnas[10]

tipoDeAcero = columnas[11:13]

index = columnas[14:21] + columnas[24:27]

log = columnas[21:24]

tipoDeFalla = columnas[27:] # Target.

#print(index) # Casilla de verificación
#print(df[index].nunique()) # Casilla de verificación

## 3.3 Investigación de Calidad
El objetivo es determinar datos únicos, duplicados, nulos y faltantes en el dataframe.

###3.3.1 Tipo de falla (Pastry, Z_Scratch, K_Scatch, Stains, Dirtiness, Bumps, Other_Faults).

In [0]:
# Investigación de Calidad: Tipo de falla.
## Verificación de variable categórica sin errores.
verificacion = (df[tipoDeFalla].sum(axis=1)==1).all()
print(f'¿Son las variables excluyentes?: {verificacion}')

## Frecuencias absolutas
frecuencias = df[tipoDeFalla].sum()
proporciones = frecuencias/df[tipoDeFalla].count()

## Datos únicos y duplicados
duplicados = []
nUnicos = []
for col in tipoDeFalla:
  duplicados.append(df[col].duplicated().sum())
  nUnicos.append(df[col].nunique())

## Tabla resumen
resumen = pd.DataFrame({
    'cantidad': frecuencias,
    'proporcion': proporciones,
    'porcentaje': proporciones * 100,
    'duplicados': duplicados,
    'únicos': nUnicos
    })

display(resumen.round(3))
print('\nTotal de registros:', frecuencias.sum())
print(f'Suma de proporciones: {proporciones.sum()} \n')

El tipo de falla corresponde a un dato binario, por lo que es correcto que de los 1941 datos, 1939 sean duplicados, y que los 2 datos únicos sean 0 y 1.
Al no tener datos nulos se verifica que los datos de estas variables están integros y sin errores.


###3.3.2 Coordenadas, Geometría y Luminosidad
- Coordenadas(X_Minimum, X_Maximum, Y_Minimum, Y_Maximum).
- Geometría (Pixels_Areas  X_Perimeter  Y_Perimeter  Steel_Plate_Thickness).
- Luminosidad(Sum_of_Luminosity  Minimum_of_Luminosity  Maximum_of_Luminosity).

In [0]:
# Investigación de Calidad: Coordenadas
## Datos únicos y duplicados de coordenadas.
duplicados = []
nUnicos = []
for col in coordenadas:
  duplicados.append(df[col].duplicated().sum())
  nUnicos.append(df[col].nunique())

## Tabla resumen Coordenadas.
resumen = pd.DataFrame({
    'duplicados': duplicados,
    'únicos': nUnicos
    }, index = coordenadas
                       )

display(resumen)
frecuencias = df[coordenadas].value_counts()
print(f'\nTotal de registros: {frecuencias.sum()} \n')


# Investigación de Calidad: Geometría
## Datos únicos y duplicados de Geometría.
duplicados = []
nUnicos = []
for col in geometria:
  duplicados.append(df[col].duplicated().sum())
  nUnicos.append(df[col].nunique())

## Tabla resumen Geometría.
resumen = pd.DataFrame({
    'duplicados': duplicados,
    'únicos': nUnicos
    }, index = geometria
                       )

display(resumen)
frecuencias = df[geometria].value_counts()
print(f'\nTotal de registros: {frecuencias.sum()} \n')


# Investigación de Calidad: Luminosidad
## Datos únicos y duplicados de Luminosidad.
duplicados = []
nUnicos = []
for col in luminosidad:
  duplicados.append(df[col].duplicated().sum())
  nUnicos.append(df[col].nunique())

## Tabla resumen Luminosidad.
resumen = pd.DataFrame({
    'duplicados': duplicados,
    'únicos': nUnicos
    }, index = luminosidad
    )

display(resumen)
frecuencias = df[luminosidad].value_counts()
print(f'\nTotal de registros: {frecuencias.sum()} \n')


Las coordenadas, la geometría y la luminosidad corresponden a datos enteros, y como cada fila corresponde a una placa, es de esperar que no todos los datos sean únicos, ya que es el conjunto de datos lo que define una placa. Aunque como hay sólo un valor duplicado de Y_Maximum, lo más probable es que haya como mucho una fila duplicada.

In [0]:
# Verificamos el potencial dato duplicado
potencialD = df[df[coordenadas[3]].duplicated(keep=False)]
print(f"Falla potencial: \n{potencialD}\n")


#Eliminamos la fila
df_limpio = df.drop(index=1776)

# Verificamos la corrección
print(f'Duplicados en dataframe original: {df[coordenadas[3]].duplicated().sum()}')
print(f'Duplicados en dataframe limpio: {df_limpio[coordenadas[3]].duplicated().sum()}')
df_limpio.shape

In [0]:
# verificar xmin < xmax y ymin < ymax
df_limpio2 = pd.Series({
    'X_Minimum < X_Maximum': (df_limpio['X_Minimum'] < df_limpio['X_Maximum']).all(),
    'Y_Minimum < Y_Maximum': (df_limpio['Y_Minimum'] < df_limpio['Y_Maximum']).all(),

# verificar pixels_areas > 0
    'Pixels_Areas > 0': (df_limpio['Pixels_Areas'] > 0).all(),

# verificar x_perimeter > 0
    'X_Perimeter > 0': (df_limpio['X_Perimeter'] > 0).all(),

# verificar y_perimeter > 0
    'Y_Perimeter > 0': (df_limpio['Y_Perimeter'] > 0).all(),

# verificar suma de luminosidad > 0 
    'Sum_of_Luminosity > 0': (df_limpio['Sum_of_Luminosity'] > 0).all(),

# verificar steel_plate_thickness > 0
    'Steel_Plate_Thickness > 0': (df_limpio['Steel_Plate_Thickness'] > 0).all(),
})


No es una fila duplicada, pero es una misma falla leída 2 veces, se eliminó una vez.

Por último, al no tener datos nulos y habiendo verificado la validez de los datos, se entiende que los datos de estas variables están integros y sin errores.

### 3.3.3 Ancho de Cinta de Inspección

In [0]:
# Investigación de Calidad: Longitud de Cinta.
## Datos únicos y duplicados de Longitud de Cinta.
duplicados = (df_limpio[longitudDeCinta].duplicated().sum())
nUnicos = (df_limpio[longitudDeCinta].nunique())

## Tabla resumen Longitud de Cinta.
resumenLC = pd.DataFrame({
    'duplicados': duplicados,
    'únicos': nUnicos
    }, index = [longitudDeCinta]
    )

display(resumenLC)
frecuencias = df_limpio[longitudDeCinta].value_counts()
print(f'\nTotal de registros: {frecuencias.sum()} \n')

Viendo el número de datos únicos, nos damos cuenta de que existen 84 longitudes de inspección en la fábrica/planta. Esto no se relaciona en forma directa con las fallas, pero permite calcular la forma del defecto a través de los índices "Outside".

###3.3.4 Índices logarítmicos

In [0]:
# Investigación de Calidad: .
## Datos únicos y duplicados de Longitud de Cinta.
duplicados = []
nUnicos = []
for col in log:
  duplicados.append(df_limpio[col].duplicated().sum())
  nUnicos.append(df_limpio[col].nunique())

## Tabla resumen Longitud de Cinta.
resumenLC = pd.DataFrame({
    'duplicados': duplicados,
    'únicos': nUnicos
    }, index = log
    )

display(resumenLC)
frecuencias = df_limpio[log].value_counts()
print(f'\nTotal de registros: {frecuencias.sum()} \n')

In [0]:
# verificar log_x_index = log10(∆x) y log_y_index = log10(∆y)
# (logaritmo base 10; atol=1e-4 tolera el redondeo de los valores guardados)
df_limpio2 = pd.Series({
    'log_x_index = log10(∆x)': np.isclose(df_limpio['Log_X_Index'], np.log10(df_limpio['X_Maximum']-df_limpio['X_Minimum']), atol=1e-4).all(),
    'log_y_index = log10(∆y)': np.isclose(df_limpio['Log_Y_Index'], np.log10(df_limpio['Y_Maximum']-df_limpio['Y_Minimum']), atol=1e-4).all(),
})
print(df_limpio2)

###3.3.5 Index

In [0]:
# Investigación de Calidad: Index
## Datos únicos y duplicados de Index.
duplicados = []
nUnicos = []
for col in index:
  duplicados.append(df_limpio[col].duplicated().sum())
  nUnicos.append(df_limpio[col].nunique())

## Evaluamos los valores máximos y mínimos de los índices, además de  los valores únicos de los OGI.
indexMax = df_limpio[index].max()
indexMin = df_limpio[index].min()

## Tabla resumen Index.
resumen = pd.DataFrame({
    'duplicados': duplicados,
    'únicos': nUnicos,
    'mínimo': indexMin,
    'máximo': indexMax
    }, index = index
                       )

display(resumen.round(3))
frecuencias = df_limpio[index].value_counts()
print(f'\nTotal de registros: {frecuencias.sum()} \n')


## Evaluamos los valores únicos de los OGI.
outsideGlobalIndex_unique = df_limpio[index[6]].unique()

## Salida
print(outsideGlobalIndex_unique)

- Outside Global Index:
  Permite describir la forma del defecto, y toma solo 3 valores:
  - 0 (Defecto alargado en X)

  - 0.5 (Defecto sin alargamiento en un eje).

  - 1 (Defecto alargado en Y).

- Orientation_Index: Coeficiente que describe la pendiente o ángulo de orientación predominante del defecto (varía entre -1 y 1).   

- Luminosity_Index: Índice que contrasta el nivel lumínico del defecto frente a la reflectividad promedio del fondo de la placa de acero (varía entre -1 y 1).

El resto de los index, al igual que las coordenadas, geometria y luminosidad, se asumen correctas mientras no hayan valores nulos.

Los demás índices varían entre 0 y 1.

###3.3.6 Tipo de Acero

In [0]:
# Investigación de Calidad: Tipo de Acero.
## Verificación de variable categórica sin errores.
verificacion = (df_limpio[tipoDeAcero].sum(axis=1)==1).all()
print(f'¿Son las variables excluyentes?: {verificacion}')

## Datos únicos y duplicados de Tipo de Acero.
duplicados = []
nUnicos = []
for col in tipoDeAcero:
  duplicados.append(df_limpio[col].duplicated().sum())
  nUnicos.append(df_limpio[col].nunique())

## Tabla resumen Tipo de Acero.
resumen = pd.DataFrame({
    'duplicados': duplicados,
    'únicos': nUnicos
    }, index = tipoDeAcero
    )

display(resumen.round(3))
frecuencias = df_limpio[tipoDeAcero].value_counts()
print(f'\nTotal de registros: {frecuencias.sum()} \n')



Dato categórico.

##3.4 Modificaciones

Como hay 2 sets de columnas categóricas, vamos a crear 2 columnas que las reemplacen.

In [0]:
# Crear TypeOfSteel y limpiar el prefijo 'TypeOfSteel_'
tipoDeMetalFila = df_limpio[tipoDeAcero].idxmax(axis=1)
df_limpio['TypeOfSteel'] = (
    tipoDeMetalFila.str.replace('TypeOfSteel_', '', regex=False)
)

# Crear TypeOfFault como variable categórica objetivo
df_limpio['TypeOfFault'] = df_limpio[tipoDeFalla].idxmax(axis=1)

# Verificar resultado
print(df_limpio[['TypeOfSteel', 'TypeOfFault']].sample(10))

df_limpio.shape

In [0]:
# Quitamos las filas innecesarias del dataframe "limpio".
df_limpio = df_limpio.drop(columns=tipoDeAcero + tipoDeFalla)

# Verificar resultado
print(df_limpio.sample(3))


#4 Análisis Univariado

In [0]:
#Has gráficos de caja para representar los outliers
#Tip 👀: En la presentación solo nombra los outliers que te llamaron la atención
#Tip 👀: Si vas a eliminar un outlier añade su respectiva justificación

##4.1 Análisis de Outliers

###4.1.1 Outliers por Coordenadas

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel 1: Solo coordenadas en X.
sns.boxplot(data=df_limpio[coordenadas[0:2]], ax=axes[0], palette=['#4C72B0', '#55A868'])
axes[0].set_title('Distribución Coordenadas Eje X')
axes[0].set_ylabel('Posición en X (píxeles)')

# Panel 2: Solo coordenadas en Y.
sns.boxplot(data=df_limpio[coordenadas[2:4]], ax=axes[1], palette=['#DD8452', '#C44E52'])
axes[1].set_title('Distribución Coordenadas Eje Y')
axes[1].set_ylabel('Posición en Y (píxeles)')

# Añadir gradillas horizontales.
axes[0].grid(axis='y')
axes[0].set_axisbelow(True)#Líneas horizontales grises que marcan el eje x

axes[1].grid(axis='y')
axes[1].set_axisbelow(True)#Líneas horizontales grises que marcan el eje x

# Graficar
plt.tight_layout()
plt.show()

# Estadística descriptiva
df_limpio[coordenadas].describe()

En el eje X no hay outliers, pero en el eje Y hay muchos outliers con una dispersión importante. Esto se debe porque estas coordenadas corresponden al marco de lectura de las placas en continuo, entonces como se trata de una linea procesadora, un marco de lectura puede haber tomado miles de placas.

Por lo que no se eliminará ningún outlier de estas mediciones

###4.1.2 Outliers (atípico) de Forma

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# Panel 1: Áreas en pixeles.
sns.boxplot(data=df_limpio[geometria[0:1]], ax=axes[0], palette=['#4C72B0'])
axes[0].set_title('Área de los pixeles con defecto')
axes[0].set_ylabel('Áreas en pixeles')

# Panel 2: Proyección del perímetro de la figura en el eje X.
sns.boxplot(data=df_limpio[geometria[1:2]], ax=axes[1], palette=['#55A868', '#189ad3'])
axes[1].set_title('Proyección del perímetro en el eje X')
axes[1].set_ylabel('Proyección del perímetro en el eje X')

# Añadir gradillas horizontales.
axes[0].grid(axis='y')
axes[0].set_axisbelow(True)

axes[1].grid(axis='y')
axes[1].set_axisbelow(True)

# Graficar
plt.tight_layout()
plt.show()

# Estadística descriptiva
df_limpio[geometria[0:2]].describe()

In [0]:
# Extraemos los outliers de las variables sobre forma y espesor para análisis
df_out = extraer_outliers_iqr(df_limpio, geometria)
print(f'''Cantidad de outliers en Pixels_Areas {df_out[0].shape}, 
      \nCantidad de outliers en X_Perimeter {df_out[1].shape}, 
      \nCantidad de outliers en Y_Perimeter {df_out[2].shape}.\n''')

# Evaluamos primero los outliers de los ejes que más llaman la atención.
print(f"Dato máximo de Pixels_Areas \n{df_out[0][df_out[0]['Pixels_Areas'] > 40000].index}\n")
print(f"Dato máximo de X_Perimeter \n{df_out[1][df_out[1]['X_Perimeter'] > 2000].index}\n")
print(f"Dato máximo de Y_Perimeter \n{df_out[2][df_out[2]['Y_Perimeter'] > 2500].index}\n")

# Revisamos el outlier
display(df_limpio.loc[[391]])

Los outliers evaluados corresponden a un mismo dato (391), asumiendo 0,5mm por pixel, la falla tendría una proyección de 5,22m en el eje x y 9,07m en el eje y. De ser 0,1mm por pixel sería de 1,04m x 1,82m. De todas formas se trata de una falla de gran complejidad ya que es 70 veces la longitud del borde en el eje x, en el eje y es sólo una vez, lo que significa que la falla tiene el grosor de un pixel en el eje y. Esto se relaciona directamente con el empty index de 0,9439 y el square index de 0.0083.

Por lo que se entiende esta falla como una rayón en zig-zag de gran tamaño.

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# Panel 1: Proyección del perímetro de la figura en el eje Y.
sns.boxplot(data=df_limpio[geometria[2:3]], ax=axes[0], palette=['#4C72B0'])
axes[0].set_title('Proyección del perímetro en el eje Y')
axes[0].set_ylabel('Píxeles en el eje Y')

# Panel 2: Espesor de la placa de acero.
sns.boxplot(data=df_limpio[geometria[3:4]], ax=axes[1], palette=['#DD8452'])
axes[1].set_title('Espesor de la placa de acero')
axes[1].set_ylabel('Espesor de la placa de acero en píxeles')

# Añadir gradillas horizontales.
axes[0].grid(axis='y')
axes[0].set_axisbelow(True)

axes[1].grid(axis='y')
axes[1].set_axisbelow(True)

plt.tight_layout()
plt.show()

# Estadística descriptiva
df_limpio[geometria[2:4]].describe()

In [0]:
# Extraemos los outliers
df_out = extraer_outliers_iqr(df_limpio, geometria)

# Observamos la cantidad de outliers de espesor.
print(f'Cantidad de outliers en Steel_Plate_Thickness \n{df_out[3].shape}\n')

# Evaluamos los outiers del espesor de las placas de acero sobre 300px.
print(f'Outliers de Steel_Plate_Thickness > 300 \n{df_out[3][df_out[3]['Steel_Plate_Thickness'] >= 300].shape}')

Como se tienen 241 datos sobre el límite superior, borrar los outliers implicaría perder muchos datos. Por lo que por falta de información sobre los objetivos de producción, no podemos determinar si estos datos atípicos son errores.

Recordar también que se trabaja con 24 grosores distintos de acero entre los 1940 datos.

Además, se debe estudiar la relación entre el grosor del acero y el tipo de falla.

###4.1.3 Outliers Luminosidad

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# Panel 1: Suma de Luminosidad.
sns.boxplot(data=df_limpio[luminosidad[0:1]], ax=axes[0], palette=['#4C72B0'])
axes[0].set_title('Luminosidad acumulada')
axes[0].set_ylabel('Escala de grises acumulada')

# Panel 2: Espesor de la placa de acero.
sns.boxplot(data=df_limpio[luminosidad[1:]], ax=axes[1], palette=['#DD8452', '#55A868'])
axes[1].set_title('Mínimo y máximo de luminosidad en un píxel')
axes[1].set_ylabel('Escala de grises')

# Añadir gradillas horizontales.
axes[0].grid(axis='y')
axes[0].set_axisbelow(True)

axes[1].grid(axis='y')
axes[1].set_axisbelow(True)

plt.tight_layout()
plt.show()

# Estadística descriptiva
df_limpio[luminosidad].describe()

La luminosidad se mide por píxel perteneciente a una falla y se encuentra en escala de grises, con 0 completamente negro y 255 completamente blanco.
En este caso se observa que el máximo registrado fue 253, y el mínimo fue 0.

### 4.1.4 Outliers Logarítmicos

Por otro lado los logaritmos entregan información del cuadro delimitador de ancho ∆x y largo ∆y que envuelve la falla detectada. Outliers aquí indicarían fallas de tamaños poco comunes en uno de los ejes de coordenadas.

In [0]:
fig, ax = plt.subplots(figsize=(6, 5))
# Solo coordenadas en X (coordenadas[0:2])
sns.boxplot(data=df_limpio[log], ax=ax, palette=['#4C72B0','#55A868', '#DD8452'])
ax.set_title('Proyección del perímetro en el eje Y')
ax.set_ylabel('Píxeles en el eje Y')

# Añadir rejillas horizontales
ax.grid(axis='y')
ax.set_axisbelow(True)

# Graficar
plt.tight_layout()
plt.show()

# Estadística descriptiva
df_limpio[log].describe()

In [0]:
# Extraemos los outliers según coordenadas para análisis
df_out = extraer_outliers_iqr(df_limpio, log)

# Evaluamos los outliers de los logaritmos de las áreas.
print(f'Outliers de LogOfAreas \n{df_out[0].index}\n')
print(f'Outliers de Log_X_Index \n{df_out[1].index}\n')
print(f'Outliers de Log_Y_Index \n{df_out[2].index}\n')

# Evaluamos el valor máximo de Logs
maxLogA = df_limpio['LogOfAreas'].idxmax()
print(f'Valor máximo de LogOfAreas: {maxLogA}\n')

maxLogX = df_limpio['Log_X_Index'].idxmax()
print(f'Valor máximo de Log_X_Index: {maxLogX}\n')

maxLogY = df_limpio['Log_Y_Index'].idxmax()
print(f'Valor máximo de Log_Y_Index: {maxLogY}\n')

# Evaluamos el valor mínimo de Log_Y y de LogOfAreas
minLogY = df_limpio['Log_Y_Index'].idxmin()
print(f'Valor mínimo de Log_Y_Index: {minLogY}\n')

minLogA = df_limpio['LogOfAreas'].idxmin()
print(f'Valor mínimo de LogOfAreas: {minLogA}\n')


In [0]:
# Auditar la fila 621, el dato de 2px x 1px.
display(df_limpio.loc[[621]])

# Auditar la fila 1098, el dato de mayor longitud horizontal que corresponde a un bump/bulto.
display(df_limpio.loc[[1098]])

# Auditar la fila 391, el dato de mayor longitud vertical.
display(df_limpio.loc[[391]])


Los outliers de Log_Y_Index preocupan más, porque 2 de ellos son de 1 pixel de altura, mientras que otra es de 20000 pixeles, lo que vendría siendo como 20m. Estos outliers por su magnitud serán descartados.

Los outliers de Log_X_Index corresponden en su mayoría a K_scatch u Other_Faults, por lo que no se trata de un error como tal sino a la naturaleza de la falla. Aunque hay un único dato que corresponde a bump, el cual debe compararse posteriormente con otros bumps y con Scratch para validar si está bien catalogado.

###4.1.5 Outliers Índices

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# Panel 1: Solo coordenadas en X (coordenadas[0:2])
sns.boxplot(data=df_limpio[index[0:1]+index[2:4]], ax=axes[0], palette=['#4C72B0', '#55A868', '#DD8452'])
axes[0].set_title('Índices de forma y posición de la bounding box')
axes[0].set_ylabel('Adimensional')

# Panel 2: Solo coordenadas en Y (coordenadas[2:4])
sns.boxplot(data=df_limpio[index[1:2]+index[4:6]], ax=axes[1], palette=['#4C72B0', '#55A868', '#DD8452'])
axes[1].set_title('Índice de forma de la falla')
axes[1].set_ylabel('Adimensional')

# Add horizontal grid
axes[0].grid(axis='y')
axes[0].set_axisbelow(True)

axes[1].grid(axis='y')
axes[1].set_axisbelow(True)

plt.tight_layout()
plt.show()

# Estadística descriptiva
df_limpio[index[0:1]+index[2:3]+index[1:2]+index[4:6]].describe()

Edges_Index indica la distancia mínima entre los bordes de la placa en el eje X y la bounding box.

Square_Index indica que tanto se asemeja la falla a un cuadrado, con 1 un cuadrado perfecto.

Outside_X_Index indica la fracción del ancho del marcho de lectura (lenght_of_conveyer) que ocupa la bounding box en el eje x.

Empty_Index indica que fracción de la caja no contiene pixeles que correspondan a la falla, con 1 todos los píxeles vacíos y 0 todos los píxeles contienen falla.

Edges_X_Index y Edges_Y_Index son índices de forma, cercano a 0 es una alta deformidad, cercano a 0,5 es una falla cuadrada, y cercano a 1 es una falla de grosor 1 píxel en ese eje.

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# Panel 1: Solo coordenadas en X (coordenadas[0:2])
sns.boxplot(data=df_limpio[index[6:7]+index[9:]], ax=axes[0], palette=['#4C72B0', '#55A868'])
axes[0].set_title('Índices')
axes[0].set_ylabel('Adimensional')

# Panel 2: Solo coordenadas en Y (coordenadas[2:4])
sns.boxplot(data=df_limpio[index[7:9]], ax=axes[1], palette=['#DD8452', '#4C72B0'])
axes[1].set_title('Espesor de la placa de acero')
axes[1].set_ylabel('Adimensional')

# Add horizontal grid
axes[0].grid(axis='y')
axes[0].set_axisbelow(True)

axes[1].grid(axis='y')
axes[1].set_axisbelow(True)

plt.tight_layout()
plt.show()

# Estadística descriptiva
df_limpio[index[6:7]+index[9:]+index[7:9]].describe()

Las distribuciones de estos índices normalizados facilitan el estudio de las formas.

Del análisis gráfico y descriptivo obtenemos que las fallas son mayormente verticales, de tamaño reducido y con baja luminosidad.

Outside_Global_Index indica cualitativamente que son mayormente verticales, y Orientation_Index cuantifica el elongamiento de las fallas, indicando que las fallas verticales tienden a tener mayor elongamiento que las horizontales, además de indicando la existencia de casos límites donde uno de los ejes es excesivamente más grande que el otro.

SigmoidOfAreas es una normalización de los valores de área entre 0 y +∞, donde si el valor del área se aproxima a 0 SigmoidOfAreas también, y si se aproxima a +∞ es cercano a 1. Esto permite analizar de mejor forma mediciones de tamaños extremadamente grandes como la falla 391, o extremadamente pequeñas como la falla 621.

Luminosity_Index indica si la falla es en promedio más oscura o más brillante. Se entiende entonces que la mayoría de las fallas son más oscuras, además de que la distribución está muy concentrada en el rango intercuartílico, en comparación con otros índices.

##4.2 Análisis Gráfico / Distribución de Fallas

###4.2.1 Distribución según tipo de falla
Queremos visualizar en forma prematura, qué tipo de falla tiene mayor incidencia.

In [0]:
## Conteo de fallas por tipo de falla.
conteo_fallas = df_limpio['TypeOfFault'].value_counts()

## Definir ejes de la gráfica
barras = conteo_fallas.index
altura = conteo_fallas.values

## Mapa de color y numeración de barras para asignación
cMap1 = plt.get_cmap("Set3")
x_ax = np.arange(len(barras))

## Gráfica de número de fallas por tipo de falla
plt.bar(barras, altura, 
        align="center", 
        color = cMap1(x_ax))
plt.title("Distribución de Clases (Target)")
plt.xlabel("Tipo de Falla")
plt.ylabel("Cantidad de Registros")
plt.show()


En orden, las fallas más comunes son K_Scatch, Bumps y Z_Scratch.

Other Faults corresponde a un cúmulo de fallas no explícitamente clasificadas, por lo que no podemos indicar si corresponde a una falla puntual con mayor incidencia que las otras.

###4.2.2 Longitud de Cinta
Queremos visualizar qué longitud de cinta produce más fallas.

In [0]:
## Conteo y ordenar duplicados por dato único
longOrdenada = contarOrdenar(df_limpio, longitudDeCinta)

## Definir ejes de la gráfica
barras = longOrdenada.index
altura = longOrdenada.values

## Mapa de color y numeración de barras para asignación
cMap2 = plt.get_cmap("viridis")
x_ax = np.linspace(0,1,len(barras))

## Gráfica de número de fallas por tipo de falla
plt.figure(figsize=(20, 8))
plt.bar(x_ax, altura, 
        color = cMap2(x_ax), 
        width=0.01)
plt.title("Cantidad de fallas por longitud de cinta")
plt.xticks(x_ax, barras, rotation=90)
plt.xlabel("Longitud de cinta")
plt.ylabel("Cantidad de Registros")
plt.show()

En base a la distribución de fallas, sabemos que los registros corresponden principalmente a las cintas entre 1352-1366, 1387, 1400, 1402 y 1686-1698.
Como no tenemos información sobre la operación sin fallas, no podemos concluir nada a partir de estos datos por si solos. Ya que, si todas las máquinas procesan el mismo número de placas en un período de tiempo t, entonces se recomendaría no utilizar las cintas con más fallas.

Pero, existe la posibilidad de que los registros de longitudes de cinta distintas a las mencionadas, correspondan a cintas que se utilizan menos. De ser este el caso, se debieran despreciar los longitudes menos utilizadas.

Además, el motivo principal para la incorporación de este dato, es para calcular los "Outside Index", con el fin de describir la forma.

Por ende, por falta de claridad y sentido físico, se despreciará este dato para el análisis.

###4.2.3 Índice de Elongamiento (Outside Global Index)

In [0]:
# Conteo y orden de duplicados por dato único de Outside Global Index (OGI)
OGI = contarOrdenar(df_limpio, index[6])

## Definir ejes de la gráfica
etiquetas_ogi = {
    0.0: "Elongamiento horizontal",
    0.5: "Sin elongamiento",
    1.0: "Elongamiento vertical",
}

cortes = OGI.index.map(etiquetas_ogi)
tamaño = OGI.values

## Colores
colores = ["#bd3939", "#399ba3", "#9addbd"]

## Gráfica de número de fallas por tipo de falla
plt.pie(
    tamaño,
    labels=cortes,
    colors=colores,
    autopct="%1.1f%%",
    pctdistance=0.75,
    wedgeprops={"linewidth": 1, "edgecolor": "white"},
)
plt.title("Proporción de fallas según elongamiento")
plt.show()

En su mayoría las fallas presentan un alargamiento en un eje.

Esto se debe a la sensibilidad de este índice, para que OGI sea 0.5, Outside_X_index y Outside_Y_Index deben ser idénticos o prácticamente idénticos según la configuración de tolerancia.

### 4.2.4 Tipo de Acero

In [0]:
## Conteo de fallas por tipo de Acero.
conteoAcero = df_limpio['TypeOfSteel'].value_counts()

## Definir ejes de la gráfica
aceros = conteoAcero.index
cantidad = conteoAcero.values

## Colores
colores = ["#bd3939",	"#399ba3"]

## Gráfica de número de fallas por tipo de falla
plt.pie(cantidad, 
        labels=aceros, 
        colors=colores,
        autopct="%1.1f%%",
        pctdistance=0.75, 
        wedgeprops = { 'linewidth' : 1, 'edgecolor' : 'white' })
plt.title("Proporción de fallas según tipo de acero")
plt.show()

Se tiene que 1164 datos corresponde a A400, y 777 corresponden a A300. La diferencia no es lo suficientemente significativa como para descargar ninguno, se debe relacionar con tipo de falla.

### 4.2.5 Distribución de espesor de acero.

In [0]:
# Graficamos las distribuciones de los 24 espesores de placas.
## Conteo y ordenar duplicados por dato único
espesorOrdenado = contarOrdenar(df_limpio,'Steel_Plate_Thickness')

## Definir ejes de la gráfica
barras = espesorOrdenado.index
altura = espesorOrdenado.values

## Mapa de color y numeración de barras para asignación
cMap2 = plt.get_cmap("viridis")
x_ax = np.arange(len(barras))

## Gráfica de número de fallas por tipo de falla
plt.figure(figsize=(15, 5))
plt.bar(x_ax, altura, 
        color = cMap2(x_ax), 
        width=1)
plt.title("Cantidad de fallas por espesor de acero")
plt.xticks(x_ax, barras, rotation=90)
plt.xlabel("Espesor de acero [mm]")
plt.ylabel("Cantidad de Registros")
plt.show()

La mayor parte de las fallas registradas, fueron en espesor de 40mm.

#5 Análisis Bivariado

##5.1 Tipo de Acero vs Tipo de Falla

In [0]:
# Posición de cada tipo de falla
# Orden fijo de tipos: garantiza el mismo orden y largo en los 3 grupos
tipos_falla = ['Pastry', 'Z_Scratch', 'K_Scratch', 'Stains', 'Dirtiness', 'Bumps', 'Other_Faults']

df_a300 = df_limpio[df_limpio['TypeOfSteel']=='A300']
conteoFallas_a300 = df_a300['TypeOfFault'].value_counts().reindex(tipos_falla, fill_value=0)

df_a400 = df_limpio[df_limpio['TypeOfSteel']=='A400']
conteoFallas_a400 = df_a400['TypeOfFault'].value_counts().reindex(tipos_falla, fill_value=0)

# Definir posición de las barras en la gráfica
r1 = [1,3,5,7,9,11,13]
r2 = [2,4,6,8,10,12,14]
r3 = r1 + r2

# Definir altura de las barras en la gráfica
altura1 = conteoFallas_a300.values
altura2 = conteoFallas_a400.values
altura = altura1 + altura2

# Definir barras
barras1 = conteoFallas_a300.index
barras2 = conteoFallas_a400.index

# Mapa de color y numeración de barras para asignación
cMap1 = plt.get_cmap("Set3")
x_ax = np.arange(len(r3))
barWidth = 0.8

# Crear gráfico de barras
plt.figure(figsize=(10, 5))
plt.bar(r1, altura1, width = barWidth, color = cMap1(6), label='A300')
plt.bar(r2, altura2, width = barWidth, color = cMap1(4), label='A400')

# Leyendas
plt.legend()

# Nombre de las variables en el eje X
plt.xticks([r + barWidth - 0.3 for r in r1], barras1)
plt.show()

Las fallas tipo K Scratch y Stains ocurren únicamente en acero A400, por lo que son buenos parámetros excluyentes, para machine learning supervisado.

Dirtiness y Pastry son predominantes en A400 en este dataset, pero no excluyentes, debe analizarse en multivariado.

Z Scratch y Bumps son predominantes en acero A300, debe analizarse en multivariado.

De Other Faults no se puede concluir nada debido a que comprende un conjunto de fallas de características desconocidas.

##5.2 Tipo de elongamiento vs Tipo de Falla

In [0]:
# Posición de cada tipo de falla
# Orden fijo de tipos: garantiza el mismo orden y largo en los 3 grupos
tipos_falla = ['Pastry', 'Z_Scratch', 'K_Scratch', 'Stains', 'Dirtiness', 'Bumps', 'Other_Faults']

df_OGI_1 = df_limpio[df_limpio['Outside_Global_Index'] == 1]
conteoFallas_OGI_1 = df_OGI_1['TypeOfFault'].value_counts().reindex(tipos_falla, fill_value=0)

df_OGI_2 = df_limpio[df_limpio['Outside_Global_Index'] == 0.5]
conteoFallas_OGI_2 = df_OGI_2['TypeOfFault'].value_counts().reindex(tipos_falla, fill_value=0)

df_OGI_3 = df_limpio[df_limpio['Outside_Global_Index'] == 0]
conteoFallas_OGI_3 = df_OGI_3['TypeOfFault'].value_counts().reindex(tipos_falla, fill_value=0)

# Definir posición de las barras en la gráfica
r1 = [1,4,7,10,13,16,19]
r2 = [2,5,8,11,14,17,20]
r3 = [3,6,9,12,15,18,21]

# Definir altura de las barras: alineadas por tipos_falla
altura1 = conteoFallas_OGI_1.values
altura2 = conteoFallas_OGI_2.values
altura3 = conteoFallas_OGI_3.values

# Mapa de color y numeración de barras para asignación
cMap1 = plt.get_cmap("Set3")

barWidth = 0.8

# Crear gráfico de barras
plt.figure(figsize=(10, 5))
plt.bar(r1, altura1, width = barWidth, color = cMap1(6), label='Vertical')
plt.bar(r2, altura2, width = barWidth, color = cMap1(4), label='Sin alargamiento')
plt.bar(r3, altura3, width = barWidth, color = cMap1(3), label='Horizontal')

# Leyendas
plt.legend()

# Nombre de las variables en el eje X (mismo orden fijo)
plt.xticks([r + barWidth + 0.15 for r in r1], tipos_falla)

plt.show()

Las fallas tipo Pastry tienen casi exclusivamente elongamientos vertical.
Las únicas fallas con mayor frecuencia de elongamiento horizontal son K Scratch y Stains.

## 5.3 Espesor de Acero vs Tipo de Falla

In [0]:
# Crear figura y contexto de Seaborn
sns.set_context("notebook", font_scale=1.15)
fig, ax = plt.subplots(figsize=(10, 5))

# Un violín por tipo de falla vs. espesor de la placa en el eje Y
sns.violinplot(
    data=df_limpio,
    x="TypeOfFault",
    y="Steel_Plate_Thickness",
    hue="TypeOfFault",
    ax=ax,
    cut=0,
    palette="Set3",
    density_norm="count",
    inner="quart",
    legend=False,
)

ax.set_title("Espesor de la placa de acero por tipo de falla", fontweight="bold")
ax.set_xlabel("Tipo de falla")
ax.set_ylabel("Espesor de la placa (mm)")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

K Scratch y Stains ocurrieron en espesores menores a 100mm, con K Scratch principalmente en espesores de 40mm

##5.4 Índices vs Tipo de falla

###5.4.1 Luminosity_Index

In [0]:
# Crear figura y contexto de Seaborn
sns.set_context("notebook", font_scale=1.15)
fig, ax = plt.subplots(figsize=(10, 5))

# Un violín por tipo de falla vs. índice de luminosidad de la placa en el eje Y
sns.violinplot(
    data=df_limpio,
    x="TypeOfFault",
    y="Luminosity_Index",
    hue="TypeOfFault",
    ax=ax,
    cut=0,
    palette="Set3",
    density_norm="count",
    inner="quart",
    legend=False,
)

ax.set_title("Índice de luminosidad promedio por tipo de falla", fontweight="bold")
ax.set_xlabel("Tipo de falla")
ax.set_ylabel("Luminosity_Index")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

Las modas del índice de luminosidad de las distintas fallas, son prácticamente idénticas, por lo que además de para diferenciar Stains, este dato no es muy útil.

### 5.4.2 Orientation_Index

In [0]:
# Crear figura y contexto de Seaborn
sns.set_context("notebook", font_scale=1.15)
fig, ax = plt.subplots(figsize=(10, 5))

# Un violín por tipo de falla vs. índice de orientación de elongación de la falla.
sns.violinplot(
    data=df_limpio,
    x="TypeOfFault",
    y="Orientation_Index",
    hue="TypeOfFault",
    ax=ax,
    cut=0,
    palette="Set3",
    density_norm="count",
    inner="quart",
    legend=False,
)

ax.set_title("Índice de orientación de elongamiento", fontweight="bold")
ax.set_xlabel("Tipo de falla")
ax.set_ylabel("Orientation_Index")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

K Scratch es multimodal, y una de sus modas es la moda más baja (moda ≈ -0.625) del conjunto de fallas. 

Pastry y Dirtiness tienen las modas más cercanas a 1.

###5.4.3 Edges_X_Index

In [0]:
# Crear figura y contexto de Seaborn
sns.set_context("notebook", font_scale=1.15)
fig, ax = plt.subplots(figsize=(10, 5))

# Un violín por tipo de falla vs. índice de deformidad horizontal de la falla.
sns.violinplot(
    data=df_limpio,
    x="TypeOfFault",
    y="Edges_X_Index",
    hue="TypeOfFault",
    ax=ax,
    cut=0,
    palette="Set3",
    density_norm="count",
    inner="quart",
    legend=False,
)

ax.set_title("Índice de deformidad horizontal de la falla", fontweight="bold")
ax.set_xlabel("Tipo de falla")
ax.set_ylabel("Edges_X_Index")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

Las fallas tipo Stains tienen la mayor deformidad horizontal de la muestra

###5.4.4 Edges_Y_Index

In [0]:
# Crear figura y contexto de Seaborn
sns.set_context("notebook", font_scale=1.15)
fig, ax = plt.subplots(figsize=(10, 5))

# Un violín por tipo de falla vs. índice de deformidad vertical de la falla.
sns.violinplot(
    data=df_limpio,
    x="TypeOfFault",
    y="Edges_Y_Index",
    hue="TypeOfFault",
    ax=ax,
    cut=0,
    palette="Set3",
    density_norm="count",
    inner="quart",
    legend=False,
)

ax.set_title("Índice de deformidad vertical de la falla", fontweight="bold")
ax.set_xlabel("Tipo de falla")
ax.set_ylabel("Edges_Y_Index")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

###5.4.5 Edges_Index

In [0]:
# Crear figura y contexto de Seaborn
sns.set_context("notebook", font_scale=1.15)
fig, ax = plt.subplots(figsize=(10, 5))

# Un violín por tipo de falla vs. Índice de cercanía de la falla a uno de los bordes laterales.
sns.violinplot(
    data=df_limpio,
    x="TypeOfFault",
    y="Edges_Index",
    hue="TypeOfFault",
    ax=ax,
    cut=0,
    palette="Set3",
    density_norm="count",
    inner="quart",
    legend=False,
)

ax.set_title("Índice de cercanía de la falla a uno de los bordes laterales", fontweight="bold")
ax.set_xlabel("Tipo de falla")
ax.set_ylabel("Edges_Index")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

###5.4.6 Empty_Index

In [0]:
# Crear figura y contexto de Seaborn
sns.set_context("notebook", font_scale=1.15)
fig, ax = plt.subplots(figsize=(10, 5))

# Un violín por tipo de falla vs. índice de píxeles vacíos en la falla.
sns.violinplot(
    data=df_limpio,
    x="TypeOfFault",
    y="Empty_Index",
    hue="TypeOfFault",
    ax=ax,
    cut=0,
    palette="Set3",
    density_norm="count",
    inner="quart",
    legend=False,
)

ax.set_title("Índice de píxeles vacíos en la falla", fontweight="bold")
ax.set_xlabel("Tipo de falla")
ax.set_ylabel("Empty_Index")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

###5.4.7 SigmoidOfAreas

In [0]:
# Crear figura y contexto de Seaborn
sns.set_context("notebook", font_scale=1.15)
fig, ax = plt.subplots(figsize=(10, 5))

# Un violín por tipo de falla vs. transformación sigmoide de las áreas.
sns.violinplot(
    data=df_limpio,
    x="TypeOfFault",
    y="SigmoidOfAreas",
    hue="TypeOfFault",
    ax=ax,
    cut=0,
    palette="Set3",
    density_norm="count",
    inner="quart",
    legend=False,
)

ax.set_title("Transformación sigmoide de las áreas", fontweight="bold")
ax.set_xlabel("Tipo de falla")
ax.set_ylabel("SigmoidOfAreas")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

###5.4.8 Square_Index

In [0]:
# Crear figura y contexto de Seaborn
sns.set_context("notebook", font_scale=1.15)
fig, ax = plt.subplots(figsize=(10, 5))

# Un violín por tipo de falla vs. transformación sigmoide de las áreas.
sns.violinplot(
    data=df_limpio,
    x="TypeOfFault",
    y="Square_Index",
    hue="TypeOfFault",
    ax=ax,
    cut=0,
    palette="Set3",
    density_norm="count",
    inner="quart",
    legend=False,
)

ax.set_title("Parentezco de la bounding box a un cuadrado", fontweight="bold")
ax.set_xlabel("Tipo de falla")
ax.set_ylabel("Square_Index")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

## 5.5 Índices Logarítmicos

###5.5.1 LogOfAreas

In [0]:
# Crear figura y contexto de Seaborn
sns.set_context("notebook", font_scale=1.15)
fig, ax = plt.subplots(figsize=(10, 5))

# Un violín por tipo de falla vs. transformación sigmoide de las áreas.
sns.violinplot(
    data=df_limpio,
    x="TypeOfFault",
    y="LogOfAreas",
    hue="TypeOfFault",
    ax=ax,
    cut=0,
    palette="Set3",
    density_norm="count",
    inner="quart",
    legend=False,
)

ax.set_title("Normalización de las áreas por logaritmo de las áreas", fontweight="bold")
ax.set_xlabel("Tipo de falla")
ax.set_ylabel("LogOfAreas")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

### 5.5.2 Log_X_Index

In [0]:
# Crear figura y contexto de Seaborn
sns.set_context("notebook", font_scale=1.15)
fig, ax = plt.subplots(figsize=(10, 5))

# Un violín por tipo de falla vs. transformación sigmoide de las áreas.
sns.violinplot(
    data=df_limpio,
    x="TypeOfFault",
    y="Log_X_Index",
    hue="TypeOfFault",
    ax=ax,
    cut=0,
    palette="Set3",
    density_norm="count",
    inner="quart",
    legend=False,
)

ax.set_title("Logaritmo de la longitud en X de la bounding box", fontweight="bold")
ax.set_xlabel("Tipo de falla")
ax.set_ylabel("Log_X_Index")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

###5.5.3 Log_Y_Index

In [0]:
# Crear figura y contexto de Seaborn
sns.set_context("notebook", font_scale=1.15)
fig, ax = plt.subplots(figsize=(10, 5))

# Un violín por tipo de falla vs. transformación sigmoide de las áreas.
sns.violinplot(
    data=df_limpio,
    x="TypeOfFault",
    y="Log_Y_Index",
    hue="TypeOfFault",
    ax=ax,
    cut=0,
    palette="Set3",
    density_norm="count",
    inner="quart",
    legend=False,
)

ax.set_title("Logaritmo de la longitud en Y de la bounding box", fontweight="bold")
ax.set_xlabel("Tipo de falla")
ax.set_ylabel("Log_Y_Index")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

Algunos puntos clave:

Aunque los rayones en Z (Z_Scratch) presentan una menor frecuencia de datos que las abolladuras (Bumps), ambas fallas comparten rangos de dispersión muy similares en las tres variables analizadas.

Las otras fallas (Other_Faults) con las abolladuras (Bumps) concentran las mayores frecuencias de registros en el conjunto de datos cercanos a 2. Y si se comparan con los rayones en Z (Z_Scratch), los cuales tienen un volumen mucho menor de frecuencia, muestran un comportamiento similar en los índices X e Y al acumular su mayor frecuencia en torno a la magnitud logarítmica de 1.

Puntos clave observando los rangos por área:

Área logarítmica: Los rayones en Z, las escamas y las abolladuras tienen rangos muy similares, concentrándose aproximadamente entre 1,2 y 4,3.

Índice X logarítmico: Acá los rangos en todas las fallas disminuyen. Y existen similitud entre las fallas de Suciedad y Manchas.

Índice Y logarítmico: Se vuelven a observar rangos similares entre los rayones en Z y las escamas, así como entre la suciedad y las otras fallas.

Haciendo un análisis específico de Cortes K y outlier (dato Nro 621)

Al analizar los diagramas de violín para las fallas de tipo "K_Scatch", la mayor concentración de datos se ubica en los siguientes intervalos:

Área logarítmica: entre 2,0 y 4,7 aproximadamente.
Índice X logarítmico: entre 1,0 y 2,7.
Índice Y logarítmico: entre 1,0 y 2,5.
Sin embargo, la observación 621 queda completamente fuera de estas zonas de concentración, registrando magnitudes extremas de 0,301 en el área logarítmica y 0 en los índices X e Y, lo que puede generar un "estiramiento" a los diagramas de violín.

Por esto, se quiere comparar los diagramas de violín de recién con otros que no contengan el dato 621, así para ver si varían o no varían visualmente.

#6 Análisis Multivariado

Vamos a comparar aquí las variables cuyas modas y cuartiles parecían diferenciar más las fallas Z Scratch y K Scratch.
- TypeOfSteel
- Steel_Plate_Thickness
- Orientation_Index
- Edges_Y_Index

In [0]:
#Filtrar solo las dos fallas y unir acero + falla en una columna
df_sub = df_limpio[df_limpio["TypeOfFault"].isin(["K_Scratch", "Z_Scratch"])].copy()

df_sub["Grupo"] = df_sub["TypeOfSteel"] + " - " + df_sub["TypeOfFault"]

colores = {
    "A300 - K_Scratch": "blue",
    "A300 - Z_Scratch": "orange",
    "A400 - K_Scratch": "green",
    "A400 - Z_Scratch": "red",
}

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection="3d")

for grupo, color in colores.items():
    datos = df_sub[df_sub["Grupo"] == grupo]
    ax.scatter(
        datos["Steel_Plate_Thickness"],
        datos["Edges_Y_Index"],
        datos["Orientation_Index"],
        color=color,
        label=grupo
    )

ax.set_title("Comparación 3D por acero y falla")
ax.set_xlabel("Steel_Plate_Thickness")
ax.set_ylabel("Edges_Y_Index")
ax.set_zlabel("Orientation_Index")
ax.legend(title="Acero - Falla")

plt.show()

Se diferencian con claridad los conjuntos de datos, ya que quedan bien agrupados en la gráfica tridimensional.

In [0]:
# Conteo exacto por grupo acero-falla
print("Conteo por TypeOfSteel y TypeOfFault:")
display(pd.crosstab(df_sub["TypeOfSteel"], df_sub["TypeOfFault"], margins=True).reset_index())

# 2. Estadísticos robustos por grupo
cols_vars = ["Steel_Plate_Thickness", "Edges_Y_Index", "Orientation_Index"]
resumen = df_sub.groupby("Grupo")[cols_vars].agg(
    ["count", "median", lambda s: s.quantile(0.75) - s.quantile(0.25)]
)
resumen.rename(columns={"<lambda_0>": "IQR"}, level=1, inplace=True)
display(resumen.reset_index())

Antes de utilizar Pearson, se deben separar las variables cuantitativas que son dicotómicas (0 y 1) con las continuas entre -1 y 1, ya que Pearson es utilizado para valores cuantitativos continuos.
Por otro lado, considerando que las relaciones de variables de coordenadas se encuentran integradas en las variables logarítmicas, para representar ambas se utilizarán sólo éstas últimas.

In [0]:
columnas_pearson=['LogOfAreas', 'Log_X_Index', 'Log_Y_Index','Pixels_Areas', 'SigmoidOfAreas','Orientation_Index','Luminosity_Index','Steel_Plate_Thickness']

corr_seleccion = df_limpio[columnas_pearson].corr(method='pearson')

plt.figure(figsize=(8, 6))
sns.heatmap(corr_seleccion, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title("Correlación de Pearson - Variables seleccionadas")
plt.show()

#7 Estadística Inferencial Test de hipótesis

Considerando las relaciones de análisis univariado, bivariado y multivariado, para lograr el objetivo principal, se hacen las afirmaciones para las hipótesis nulas y alternativas:

1. Al seleccionar cualquier muestra de acero de A300, las fallas con espesor de placa cercano a 70mm y con Índice de bordes en Y cercanas a 1, tienen mayor probabilidad de ser Fallas Z-S.

2. Al seleccionar cualquier muestra de acero de A400, las fallas con espesor de placa cercano a  40mm y con Índice de bordes en Y cercanos a 0, tienen mayor probabilidad de ser Fallas K-S.


¿Por qué 70 mm y 40 mm?

Se obtuvieron estos valores al observar los intercuartiles de las fallas de K-S y Z-S, ambas fallas, contenían intercuartiles de 0, es decir: Q1, mediana y Q3 coincidian con los valores 70mm y 40mm.

Por lo tanto, las hipótesis de las afirmaciones serán:
1.

H0= En muestras de acero A300, tener un espesor de placa cercano a 70 mm y un Índice de bordes en Y cercano a 1 no aumenta la probabilidad de presentar una falla Z-S

H1= En muestras de acero A300, tener un espesor de placa cercano a 70 mm y un Índice de bordes en Y cercano a 1 aumenta la probabilidad de presentar una falla Z-S.


2.

H0= En muestras de acero A400, tener un espesor de placa cercano a 40 mm y un Índice de bordes en Y entre 0,4 y 0  mm no aumenta la probabilidad de presentar una falla K-S.

H1=En muestras de acero A400, tener un espesor de placa cercano a 40 mm y un Índice de bordes en Y centre 0,4 y 0  mm aumenta la probabilidad de presentar una falla K-S.


###7.1 Test ______

#8 Conclusión

#9 Referencias